# 7 Ablation Preprocessing

Encode all 2000 files with the six AMSTE ablation variants.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

"""
Batch AMSTE-Ablation Spike Encoding Pipeline for DAS Microseismic Classification
================================================================================
Processes all SEGY files (Event + Noise) through the 6 AMSTE ablation
variants and saves per-variant .npy files ready for SNN training.

Ablation variants (leave-one-out):
    [V1] Basic         — all 4 AMSTE components removed (lower bound)
    [V2] No-MAD        — global threshold replaces per-channel θ_c
    [V3] No-Polarity   — |Δx| only (no +/− split)
    [V4] No-MultiScale — single lag (lag = 1) only
    [V5] No-Spatial    — no spatial coherence gate
    [V6] Full AMSTE    — proposed full method (alpha, lags, min_votes, ws, thr_s)

All 6 variants take the same signed-normalised input [-1, +1] and produce
the same shape (n_traces, n_samples) float32 binary spike maps. Any
classification-metric difference between variants in notebook 4 is therefore
attributable to the ablated component, not to retuning or input differences.

AMSTE parameters (from confirmed multifile sweep, notebook 2.2):
    alpha     = 3.0
    lags      = [1, 3, 8]        (0.5, 1.5, 4.0 ms)
    min_votes = 2
    ws        = 16
    thr_s     = 0.5

Output structure:
    OUTPUT_ROOT/
    ├── Basic/           file_00001.npy ...
    ├── No-MAD/
    ├── No-Polarity/
    ├── No-MultiScale/
    ├── No-Spatial/
    ├── Full_AMSTE/
    ├── labels.csv       (file_id, label, class_name, filename, n_traces, n_samples, dt_ms)
    └── batch_log.txt

Each .npy: float32 spike array (n_traces, n_samples) — binary {0, 1}.

Resume support: re-running with resume=True skips files already present
                in ALL six variant directories.
"""

import segyio
import numpy as np
from scipy.signal  import butter, filtfilt
from scipy.ndimage import maximum_filter1d, minimum_filter1d
import os
import csv
import time
import traceback

# =============================================================================
# PATHS
# =============================================================================
EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
OUTPUT_ROOT = config.ENCODED_ABLATION_DIR

# =============================================================================
# PREPROCESSING
# =============================================================================
FORCE_DT_MS      = 0.5    # FRISCO FORGE 2024, 2 kHz DAS interrogator
FILTER_LOW_HZ    = 50
FILTER_HIGH_HZ   = 250
WINDOW_START_MS  = 0
WINDOW_END_MS    = 1000

# =============================================================================
# AMSTE PARAMETERS — from confirmed multifile sweep (notebook 2.2)
# =============================================================================
AMSTE_ALPHA     = 3.0          # MAD multiplier for per-channel threshold
AMSTE_LAGS      = [1, 3, 8]    # temporal scales: 0.5, 1.5, 4.0 ms
AMSTE_MIN_VOTES = 2            # majority vote (≥2/3 lags must agree)
AMSTE_WS        = 16           # spatial window: 16 channels (64 m aperture)
AMSTE_THR_S     = 0.5          # spatial amplitude gate threshold

# ── Variant names — used as output directory names ───────────────────────────
# Filesystem-safe names (no spaces, dashes replaced with underscores).
VARIANT_NAMES = [
    'Basic',
    'No_MAD',
    'No_Polarity',
    'No_MultiScale',
    'No_Spatial',
    'Full_AMSTE',
]


# =============================================================================
# PREPROCESSING FUNCTIONS
# =============================================================================
def load_segy_quiet(filepath):
    with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
        data = np.array([f.trace[i] for i in range(len(f.trace))])
        try:
            dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
            if not (0.001 <= dt_ms <= 10.0):
                dt_ms = FORCE_DT_MS
        except Exception:
            dt_ms = FORCE_DT_MS
    return data, dt_ms


def bandpass_filter(data, dt_ms):
    fs   = 1000.0 / dt_ms
    nyq  = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ  / nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ / nyq, 1e-6, 0.99)],
                  btype='band')
    out  = np.zeros_like(data)
    for i in range(data.shape[0]):
        out[i] = filtfilt(b, a, data[i])
    return out


def extract_time_window(data, dt_ms):
    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(data.shape[1], int(WINDOW_END_MS / dt_ms))
    return data[:, s:e] if s < e else data


def normalize_signed(data):
    """
    Signed normalisation per trace → [-1, +1].
    AMSTE requires signed input so that bidirectional polarity voting works:
    a -0.8 strain excursion must remain at -0.8 (visible to neg-vote), not
    map to 0.1 as it would under unit [0, 1] normalisation.
    """
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[0]):
        t = data[i] - np.mean(data[i])
        m = np.max(np.abs(t))
        out[i] = t / m if m > 0 else t
    return out


def preprocess(filepath):
    """
    Full preprocessing pipeline — bandpass, time-window, signed normalisation.
    Returns norm_signed (n_traces, n_samples) and dt_ms.
    """
    data, dt_ms = load_segy_quiet(filepath)
    filtered    = bandpass_filter(data, dt_ms)
    filtered    = extract_time_window(filtered, dt_ms)
    norm_signed = normalize_signed(filtered)
    return norm_signed, dt_ms


# =============================================================================
# AMSTE ABLATION VARIANTS (V1 … V6, fully vectorised)
# =============================================================================
# All variants take signed-normalised input in [-1, +1] and return a binary
# {0, 1} float32 spike map of the same shape. Variants share parameters
# where applicable, so any output difference is attributable to the ablated
# component, not to retuning.
# -----------------------------------------------------------------------------

def _per_channel_mad_threshold(data, alpha):
    """Per-channel MAD threshold:  θ_c = α · 1.4826 · MAD_c    (n_tr, 1)."""
    med_c = np.median(data, axis=1, keepdims=True)
    mad_c = np.median(np.abs(data - med_c), axis=1, keepdims=True)
    return np.maximum(alpha * 1.4826 * mad_c, 1e-9)

def _global_mad_threshold(data, alpha):
    """Global MAD threshold: one scalar shared across all channels."""
    med = np.median(data)
    mad = np.median(np.abs(data - med))
    return float(max(alpha * 1.4826 * mad, 1e-9))

def _spatial_da(data, ws):
    """Spatial amplitude-spread: max(|x|) − min(|x|) over a ws-channel window."""
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    return spat_max - spat_min


def _basic_change(data, dt_ms):
    """[V1] None of the 4 AMSTE components.
        spike[c, t] = 1  if  |x[c, t] − x[c, t−1]| > θ_global
    Single lag, no polarity, no spatial gate, scalar global threshold."""
    theta = _global_mad_threshold(data, AMSTE_ALPHA)
    diff  = np.abs(np.diff(data, axis=1, prepend=data[:, :1]))
    return (diff > theta).astype(np.float32)


def _amste_no_mad(data, dt_ms):
    """[V2] Global θ replaces per-channel θ_c.
    Polarity, multi-scale voting, and spatial gate are kept."""
    n_tr, n_smp = data.shape
    theta = _global_mad_threshold(data, AMSTE_ALPHA)
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in AMSTE_LAGS:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta).astype(np.int32)
    candidate = (pos_votes >= AMSTE_MIN_VOTES) | (neg_votes >= AMSTE_MIN_VOTES)
    da_s = _spatial_da(data, AMSTE_WS)
    return (candidate & (da_s > AMSTE_THR_S)).astype(np.float32)


def _amste_no_polarity(data, dt_ms):
    """[V3] |Δx| only (no +/− split).
    MAD threshold, multi-scale voting, and spatial gate are kept."""
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, AMSTE_ALPHA)
    votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in AMSTE_LAGS:
        diff_l = np.abs(data[:, lag:] - data[:, :-lag])
        votes[:, lag:] += (diff_l > theta_c).astype(np.int32)
    candidate = (votes >= AMSTE_MIN_VOTES)
    da_s = _spatial_da(data, AMSTE_WS)
    return (candidate & (da_s > AMSTE_THR_S)).astype(np.float32)


def _amste_no_multiscale(data, dt_ms):
    """[V4] Single lag = 1 only (no multi-scale majority vote).
    MAD threshold, polarity, and spatial gate are kept."""
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, AMSTE_ALPHA)
    diff_l = data[:, 1:] - data[:, :-1]
    pos = np.zeros((n_tr, n_smp), dtype=bool)
    neg = np.zeros((n_tr, n_smp), dtype=bool)
    pos[:, 1:] = (diff_l >  theta_c)
    neg[:, 1:] = (diff_l < -theta_c)
    candidate = (pos | neg)
    da_s = _spatial_da(data, AMSTE_WS)
    return (candidate & (da_s > AMSTE_THR_S)).astype(np.float32)


def _amste_no_spatial(data, dt_ms):
    """[V5] No neighbourhood amplitude-spread check.
    MAD threshold, polarity, and multi-scale voting are kept."""
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, AMSTE_ALPHA)
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in AMSTE_LAGS:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)
    candidate = (pos_votes >= AMSTE_MIN_VOTES) | (neg_votes >= AMSTE_MIN_VOTES)
    return candidate.astype(np.float32)


def _amste_full(data, dt_ms):
    """
    [V6] Full AMSTE — all four components active.

    STEP 1  Per-channel MAD threshold       θ_c = α · 1.4826 · MAD_c
    STEP 2  Bidirectional polarity at multiple lags (per-channel)
    STEP 3  Multi-scale majority vote per polarity (≥ min_votes)
    STEP 4  Spatial coherence gate          da_s > θ_s

    Reference design basis:
        - Channel-adaptive threshold: PhaseNet-DAS (Nature Comms 2023)
        - Polarity coding: Neural Coding in SNNs (Frontiers Neurosci 2021)
        - Multi-scale delta:  sensor SNN evaluation (arXiv 2407.09260, 2024)
        - Spatial coherence:  extends ST-MW (Petro et al. TNNLS 2020)

    OUTPUT: binary {0, 1} — drop-in compatible with notebook 4 SNN training.
    """
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, AMSTE_ALPHA)
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in AMSTE_LAGS:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)
    candidate = (pos_votes >= AMSTE_MIN_VOTES) | (neg_votes >= AMSTE_MIN_VOTES)
    da_s = _spatial_da(data, AMSTE_WS)
    return (candidate & (da_s > AMSTE_THR_S)).astype(np.float32)


# =============================================================================
# ENCODE ALL 6 ABLATION VARIANTS
# =============================================================================
def encode_all_variants(norm_signed, dt_ms):
    """
    Run all 6 AMSTE ablation variants on the same signed-normalised input.
    Returns a dict keyed by variant directory name → float32 spike array.
    """
    return {
        'Basic':         _basic_change       (norm_signed, dt_ms),
        'No_MAD':        _amste_no_mad       (norm_signed, dt_ms),
        'No_Polarity':   _amste_no_polarity  (norm_signed, dt_ms),
        'No_MultiScale': _amste_no_multiscale(norm_signed, dt_ms),
        'No_Spatial':    _amste_no_spatial   (norm_signed, dt_ms),
        'Full_AMSTE':    _amste_full         (norm_signed, dt_ms),
    }


# =============================================================================
# BATCH PIPELINE
# =============================================================================
def collect_files(event_dir, noise_dir):
    files = []
    idx   = 0
    for fname in sorted(f for f in os.listdir(event_dir)
                        if f.lower().endswith(('.sgy', '.segy'))):
        idx += 1
        files.append((os.path.join(event_dir, fname), 1,
                      f'file_{idx:05d}', fname))
    n_events = idx

    for fname in sorted(f for f in os.listdir(noise_dir)
                        if f.lower().endswith(('.sgy', '.segy'))):
        idx += 1
        files.append((os.path.join(noise_dir, fname), 0,
                      f'file_{idx:05d}', fname))
    n_noise = idx - n_events
    return files, n_events, n_noise


def get_processed_ids(variant_dirs):
    """File IDs that have an .npy file present in EVERY variant directory."""
    sets = []
    for name in VARIANT_NAMES:
        d = variant_dirs[name]
        sets.append({os.path.splitext(f)[0]
                     for f in os.listdir(d) if f.endswith('.npy')}
                    if os.path.exists(d) else set())
    return sets[0].intersection(*sets[1:]) if sets else set()


def run_batch(resume=False):
    print("=" * 78)
    print("BATCH AMSTE-ABLATION SPIKE ENCODING PIPELINE — 6 VARIANTS")
    print(f"  Preprocessing: BP {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz | "
          f"Win {WINDOW_START_MS}-{WINDOW_END_MS} ms")
    print(f"  Normalisation: signed [-1, +1] for all variants")
    print(f"  AMSTE params:  alpha={AMSTE_ALPHA}, lags={AMSTE_LAGS}, "
          f"min_votes={AMSTE_MIN_VOTES}, ws={AMSTE_WS}, thr_s={AMSTE_THR_S}")
    print(f"  Event:   {EVENT_DIR}")
    print(f"  Noise:   {NOISE_DIR}")
    print(f"  Output:  {OUTPUT_ROOT}")
    print(f"  Resume:  {resume}")
    print(f"  Variants: " + " | ".join(VARIANT_NAMES))
    print("=" * 78)

    # Create output directories ────────────────────────────────────────────
    variant_dirs = {}
    for name in VARIANT_NAMES:
        d = os.path.join(OUTPUT_ROOT, name)
        os.makedirs(d, exist_ok=True)
        variant_dirs[name] = d

    # Collect files ────────────────────────────────────────────────────────
    print("\nScanning directories...")
    files, n_events, n_noise = collect_files(EVENT_DIR, NOISE_DIR)
    n_total = len(files)
    print(f"  Found: {n_events} event + {n_noise} noise = {n_total} files")
    if n_total == 0:
        raise RuntimeError("No SEGY files found. Check EVENT_DIR and NOISE_DIR.")

    # Resume support ───────────────────────────────────────────────────────
    skip_ids = get_processed_ids(variant_dirs) if resume else set()
    if resume:
        print(f"  Resume: {len(skip_ids)} done, "
              f"{n_total - len(skip_ids)} remaining")

    labels_path = os.path.join(OUTPUT_ROOT, 'labels.csv')
    log_path    = os.path.join(OUTPUT_ROOT, 'batch_log.txt')
    csv_mode    = 'a' if resume and os.path.exists(labels_path) else 'w'
    log_mode    = 'a' if resume else 'w'

    with open(labels_path, csv_mode, newline='') as csv_file, \
         open(log_path,    log_mode)               as log_file:

        writer = csv.writer(csv_file)
        if csv_mode == 'w':
            writer.writerow(['file_id', 'label', 'class_name',
                             'original_filename', 'n_traces',
                             'n_samples', 'dt_ms'])

        log_file.write(f"\n{'='*60}\n")
        log_file.write(f"Started: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"Files: {n_total} | Resume: {resume}\n")
        log_file.write(f"Variants: {', '.join(VARIANT_NAMES)}\n")
        log_file.write(f"AMSTE: alpha={AMSTE_ALPHA} lags={AMSTE_LAGS} "
                       f"min_votes={AMSTE_MIN_VOTES} ws={AMSTE_WS} "
                       f"thr_s={AMSTE_THR_S}\n")
        log_file.write(f"{'='*60}\n\n")

        t_start   = time.time()
        n_done    = 0
        n_skipped = 0
        n_errors  = 0
        errors    = []

        for filepath, label, file_id, orig_name in files:

            if file_id in skip_ids:
                n_skipped += 1
                continue

            class_name = 'event' if label == 1 else 'noise'

            try:
                norm_signed, dt_ms = preprocess(filepath)
                n_tr, n_smp = norm_signed.shape

                encodings = encode_all_variants(norm_signed, dt_ms)

                for name in VARIANT_NAMES:
                    np.save(
                        os.path.join(variant_dirs[name], f'{file_id}.npy'),
                        encodings[name])

                writer.writerow([file_id, label, class_name,
                                 orig_name, n_tr, n_smp, dt_ms])
                csv_file.flush()
                n_done += 1

                elapsed   = time.time() - t_start
                rate      = n_done / elapsed if elapsed > 0 else 0
                remaining = n_total - n_skipped - n_done - n_errors
                eta       = remaining / rate if rate > 0 else 0

                if n_done % 100 == 0 or n_done <= 5:
                    print(f"  [{n_done+n_skipped+n_errors:>6d}/{n_total}]  "
                          f"{file_id}  {class_name:5s}  "
                          f"({n_tr}×{n_smp})  "
                          f"{rate:.1f} f/s  "
                          f"ETA {eta/60:.0f} min")

            except Exception as ex:
                n_errors += 1
                msg = f"ERROR [{file_id}] {orig_name}: {ex}"
                errors.append(msg)
                log_file.write(msg + "\n")
                log_file.write(traceback.format_exc() + "\n")
                log_file.flush()
                if n_errors <= 10:
                    print(f"    {msg}")
                elif n_errors == 11:
                    print("    Further errors suppressed — see batch_log.txt")

        elapsed = time.time() - t_start
        print(f"\n{'='*78}")
        print(f"BATCH COMPLETE")
        print(f"  Processed : {n_done}")
        print(f"  Skipped   : {n_skipped} (resume)")
        print(f"  Errors    : {n_errors}")
        if elapsed > 0 and n_done > 0:
            print(f"  Time      : {elapsed/60:.1f} min  "
                  f"({n_done/elapsed:.1f} f/s)")
        print(f"  Labels    : {labels_path}")
        print(f"  Log       : {log_path}")
        print(f"{'='*78}")

        log_file.write(f"\nCompleted: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"Done={n_done} Skip={n_skipped} Err={n_errors} "
                       f"Time={elapsed:.0f}s\n")
        if errors:
            log_file.write(
                "\nErrors:\n" + "\n".join(f"  {e}" for e in errors) + "\n")

    # Output verification ──────────────────────────────────────────────────
    print("\nOutput verification:")
    for name in VARIANT_NAMES:
        count = len([f for f in os.listdir(variant_dirs[name])
                     if f.endswith('.npy')])
        print(f"  {name:14s}  {count:>6d} files")

    # Sample sparsity check (first non-skipped file) ───────────────────────
    sample = next((f for f in files if f[2] not in skip_ids), None)
    if sample and n_done > 0:
        sample_id = sample[2]
        print(f"\nSample sparsity check ({sample_id}):")
        for name in VARIANT_NAMES:
            fp = os.path.join(variant_dirs[name], f'{sample_id}.npy')
            if os.path.exists(fp):
                arr = np.load(fp)
                sp  = float(np.abs(arr).mean()) * 100
                print(f"  {name:14s}  {sp:.3f}%")


# =============================================================================
# DATASET LOADER UTILITIES (used by notebook 4)
# =============================================================================
def load_encoded_dataset(output_root, variant_name, max_files=None):
    """
    Load spike arrays + labels for ONE ablation variant.
    Returns:
        X        — (N, n_traces, n_samples) float32
        y        — (N,) int   (1 = event, 0 = noise)
        file_ids — list[str]
    """
    import pandas as pd
    df = pd.read_csv(os.path.join(output_root, 'labels.csv'))
    if max_files:
        df = df.head(max_files)
    enc_dir = os.path.join(output_root, variant_name)
    X, y, ids = [], [], []
    for _, row in df.iterrows():
        fp = os.path.join(enc_dir, f"{row['file_id']}.npy")
        if os.path.exists(fp):
            X.append(np.load(fp))
            y.append(row['label'])
            ids.append(row['file_id'])
    if not X:
        raise RuntimeError(f"No files loaded for variant '{variant_name}'")
    X = np.stack(X, axis=0)
    y = np.array(y, dtype=int)
    print(f"Loaded {variant_name}: X={X.shape}  "
          f"events={np.sum(y==1)}  noise={np.sum(y==0)}")
    return X, y, ids


def load_multi_variant_dataset(output_root, variant_names=None, max_files=None):
    """
    Load spike arrays from multiple ablation variants as separate channels.
    Useful for batched evaluation in notebook 4 — one DataLoader per variant,
    or for stacking variants as channels in a multi-stream architecture.

    Returns:
        X — (N, n_variants, n_traces, n_samples) float32
        y — (N,) int
        variant_names — list[str]
    """
    import pandas as pd
    if variant_names is None:
        variant_names = VARIANT_NAMES
    df = pd.read_csv(os.path.join(output_root, 'labels.csv'))
    if max_files:
        df = df.head(max_files)
    all_X = []
    for name in variant_names:
        var_dir = os.path.join(output_root, name)
        arrs = []
        for _, row in df.iterrows():
            fp = os.path.join(var_dir, f"{row['file_id']}.npy")
            arrs.append(np.load(fp) if os.path.exists(fp)
                        else np.zeros((1, 1), dtype=np.float32))
        all_X.append(np.stack(arrs, axis=0))
    X = np.stack(all_X, axis=1)
    y = df['label'].values.astype(int)
    print(f"Loaded {len(variant_names)} variants: X={X.shape}  "
          f"events={np.sum(y==1)}  noise={np.sum(y==0)}")
    return X, y, variant_names


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    # Set resume=True to continue after a crash without reprocessing done files.
    # Resume considers a file "done" only when it exists in all 6 variant
    # directories, so partial encodes will be redone safely.
    run_batch(resume=False)
